# Lead Conversion Prediction project
#### Predict whether a lead will convert into a customer or not.

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [39]:
df = pd.read_csv("data/Leads.csv")

In [40]:
df.columns

Index(['Prospect ID', 'Lead Number', 'Lead Origin', 'Lead Source',
       'Do Not Email', 'Do Not Call', 'Converted', 'TotalVisits',
       'Total Time Spent on Website', 'Page Views Per Visit', 'Last Activity',
       'Country', 'Specialization', 'How did you hear about X Education',
       'What is your current occupation',
       'What matters most to you in choosing a course', 'Search', 'Magazine',
       'Newspaper Article', 'X Education Forums', 'Newspaper',
       'Digital Advertisement', 'Through Recommendations',
       'Receive More Updates About Our Courses', 'Tags', 'Lead Quality',
       'Update me on Supply Chain Content', 'Get updates on DM Content',
       'Lead Profile', 'City', 'Asymmetrique Activity Index',
       'Asymmetrique Profile Index', 'Asymmetrique Activity Score',
       'Asymmetrique Profile Score',
       'I agree to pay the amount through cheque',
       'A free copy of Mastering The Interview', 'Last Notable Activity'],
      dtype='object')

# Data Preprocessing

In [41]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
numerical_columns = df.select_dtypes(include=['number']).columns.tolist()

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [42]:
df.head()

,prospect_id,lead_number,lead_origin,lead_source,do_not_email,do_not_call,converted,totalvisits,total_time_spent_on_website,page_views_per_visit,...,get_updates_on_dm_content,lead_profile,city,asymmetrique_activity_index,asymmetrique_profile_index,asymmetrique_activity_score,asymmetrique_profile_score,i_agree_to_pay_the_amount_through_cheque,a_free_copy_of_mastering_the_interview,last_notable_activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,api,olark_chat,no,no,0,0.0,0,0.0,...,no,select,select,02.medium,02.medium,15.0,15.0,no,no,modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,api,organic_search,no,no,0,5.0,674,2.5,...,no,select,select,02.medium,02.medium,15.0,15.0,no,no,email_opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,landing_page_submission,direct_traffic,no,no,1,2.0,1532,2.0,...,no,potential_lead,mumbai,02.medium,01.high,14.0,20.0,no,yes,email_opened
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,landing_page_submission,direct_traffic,no,no,0,1.0,305,1.0,...,no,select,mumbai,02.medium,01.high,13.0,17.0,no,no,modified
4,3256f628-e534-4826-9d63-4a8b88782852,660681,landing_page_submission,google,no,no,1,2.0,1428,1.0,...,no,select,mumbai,02.medium,01.high,15.0,18.0,no,no,modified


In [43]:
df.dtypes

prospect_id                                       object
lead_number                                        int64
lead_origin                                       object
lead_source                                       object
do_not_email                                      object
do_not_call                                       object
converted                                          int64
totalvisits                                      float64
total_time_spent_on_website                        int64
page_views_per_visit                             float64
last_activity                                     object
country                                           object
specialization                                    object
how_did_you_hear_about_x_education                object
what_is_your_current_occupation                   object
what_matters_most_to_you_in_choosing_a_course     object
search                                            object
magazine                       

In [44]:
df.isna().sum()

prospect_id                                         0
lead_number                                         0
lead_origin                                         0
lead_source                                        36
do_not_email                                        0
do_not_call                                         0
converted                                           0
totalvisits                                       137
total_time_spent_on_website                         0
page_views_per_visit                              137
last_activity                                     103
country                                          2461
specialization                                   1438
how_did_you_hear_about_x_education               2207
what_is_your_current_occupation                  2690
what_matters_most_to_you_in_choosing_a_course    2709
search                                              0
magazine                                            0
newspaper_article           

#### 

#### Some libraries handle missings natively (e.g., XGBoost/LightGBM/CatBoost). Many scikit-learn estimators do not. If you’re using LogisticRegression, SVC, KNN, etc., you must impute.

#### In real projects, prefer these defaults:

- Categorical (object): fill with a string like "NA" (creates its own category) or use most frequent via SimpleImputer(strategy="most_frequent").

- Numeric (int/float): fill with median (robust to outliers). Using 0 is okay only if 0 is a plausible value; otherwise it can bias the model. Add a missing-indicator if the fact that it was missing might be informative

In [45]:
columns = df.columns.tolist()

for c in columns:
    if c in categorical_columns:
        # fill categorical missings with a token (or use df[c].mode()[0] for most frequent)
        df[c].fillna('NA')
    elif c in numerical_columns:
        # fill numeric missings with 0 (or df[c].median())
        df[c].fillna(0)



# Setting up the validation framework
#### Perform the train/validation/test split with Scikit-Learn

In [46]:
from sklearn.model_selection import train_test_split

In [47]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [48]:
len(df_train), len(df_val), len(df_test)

(5544, 1848, 1848)

In [55]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_full_train = df_full_train.reset_index(drop=True)

In [50]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

# EDA
- Check missing values
- Look at the target variable (churn)
- Look at numerical and categorical variables

## NB: Check missing values any time, but do the actual imputation after splitting, using rules derived from the training data

In [51]:
df_full_train.converted.value_counts(normalize=True) # the class distribution of converted as fractions that sum to 1

converted
0    0.616883
1    0.383117
Name: proportion, dtype: float64

In [52]:
df_full_train['converted'].mean()

np.float64(0.38311688311688313)

In [53]:
df_full_train[categorical_columns].nunique() # How many distinct values
# Note the columns with 1 unique value, they carry no information so you don't need to use them.

prospect_id                                      7392
lead_origin                                         5
lead_source                                        19
do_not_email                                        2
do_not_call                                         2
last_activity                                      16
country                                            36
specialization                                     19
how_did_you_hear_about_x_education                 10
what_is_your_current_occupation                     6
what_matters_most_to_you_in_choosing_a_course       2
search                                              2
magazine                                            1
newspaper_article                                   2
x_education_forums                                  1
newspaper                                           2
digital_advertisement                               2
through_recommendations                             2
receive_more_updates_about_o

# Feature importance: convertion rate and risk ratio

### Conversion rate

In [56]:
df_full_train.head()

,prospect_id,lead_number,lead_origin,lead_source,do_not_email,do_not_call,converted,totalvisits,total_time_spent_on_website,page_views_per_visit,...,get_updates_on_dm_content,lead_profile,city,asymmetrique_activity_index,asymmetrique_profile_index,asymmetrique_activity_score,asymmetrique_profile_score,i_agree_to_pay_the_amount_through_cheque,a_free_copy_of_mastering_the_interview,last_notable_activity
0,e00e7924-260f-4ad7-8ad5-a246813affc0,609012,landing_page_submission,google,no,no,0,4.0,239,4.0,...,no,select,mumbai,NaN,NaN,NaN,NaN,no,no,email_opened
1,db7df067-c985-4d24-80af-c72990c68904,634813,lead_add_form,reference,no,no,1,0.0,0,0.0,...,no,potential_lead,thane_&_outskirts,02.medium,01.high,15.0,19.0,no,no,sms_sent
2,1ab4347a-252d-4d68-b178-6a924cc08c19,627932,landing_page_submission,direct_traffic,no,no,0,2.0,271,2.0,...,no,NaN,other_cities_of_maharashtra,NaN,NaN,NaN,NaN,no,yes,email_opened
3,930ff3a2-aa35-4a4d-9f27-04423d302af4,650147,api,referral_sites,no,no,0,8.0,51,4.0,...,no,select,select,02.medium,02.medium,15.0,13.0,no,no,modified
4,6b142314-ab04-4b05-9a57-90105cb2b91a,600800,landing_page_submission,direct_traffic,no,no,0,1.0,95,1.0,...,no,NaN,mumbai,NaN,NaN,NaN,NaN,no,yes,modified
